In [1]:
#pickle notebook, function, add my words to notebook
import pickle
import pandas as pd
import numpy as np
import warnings
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score

In [2]:
# To load object use pickle.load()
imp = pickle.load(open("SimpleImputer.pkl","rb"))

min_max_scaler = pickle.load(open("min_max_scaler.pkl","rb"))

#pickle for the best model
brfc_model = pickle.load(open("brfc_model.pkl","rb"))

In [3]:
# supress warnings
warnings.simplefilter('ignore')

# load test dataset; replace file path with location of test file
diabetes_data = pd.read_csv('/home/jovyan/work/midtermaters/Homework 4/PimaIndiansDiabetes.csv')
ddf = diabetes_data.copy()


In [4]:
def prep(ddf):
    #subset-all features
    ddfsub = ddf[["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]]
    ddfsub2 = ddfsub.copy()

    # 1) Define “invalid zero” columns
    zero_missing = [
        "Glucose",          # cannot have 0 mg/dL
        "BloodPressure",    # cannot have  0 mm Hg
        "SkinThickness",    # cannot have  0 mm skin fold
        "Insulin",          # cannot have  0 μU/ml
        "BMI"               # cannot have  0 kg/m²
    ]

# 2) Convert placeholder zeros to NaN
    ddfsub2.loc[:, zero_missing] = ddfsub2.loc[:, zero_missing].replace(0, np.nan)
    # ddfsub2 = ddfsub2[zero_missing]
    ddfsub2[zero_missing] = imp.transform(ddfsub2[zero_missing]) #impute missing values (imp fit on previous dataset)
    # df_imputed = pd.DataFrame(imputed_arr, columns= ddfsub.columns, index=ddfsub.index) # convert arr to df, restore original column names and index
    ddfsub2 = min_max_scaler.transform(ddfsub2) #scale with min_max_scaler (fit on prev dataset)
    ddfsub2 = pd.DataFrame(ddfsub2, columns=ddfsub.columns) # convert arr to df
    return ddfsub2

In [5]:
print("Imputer was fitted on:", imp.feature_names_in_)
print("Scaler was fitted on:", min_max_scaler.feature_names_in_)

Imputer was fitted on: ['Glucose' 'BloodPressure' 'SkinThickness' 'Insulin' 'BMI']
Scaler was fitted on: ['Pregnancies' 'Glucose' 'BloodPressure' 'SkinThickness' 'Insulin' 'BMI'
 'DiabetesPedigreeFunction' 'Age']


In [9]:
df_scaled = prep(ddf)
print(ddf)
print(df_scaled)
prob = brfc_model.predict_proba(df_scaled)[:,1]
y = ddf["Outcome"]
X = df_scaled
feature_results = [roc_auc_score(y, brfc_model.predict_proba(X)[:,1])]

     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0              6      148             72             35        0  33.6   
1              1       85             66             29        0  26.6   
2              8      183             64              0        0  23.3   
3              1       89             66             23       94  28.1   
4              0      137             40             35      168  43.1   
..           ...      ...            ...            ...      ...   ...   
663            9      145             80             46      130  37.9   
664            6      115             60             39        0  33.7   
665            1      112             80             45      132  34.8   
666            4      145             82             18        0  32.5   
667           10      111             70             27        0  27.5   

     DiabetesPedigreeFunction  Age  Outcome  
0                       0.627   50        1  
1                  

In [8]:
feature_results

[0.8620761389640108]